# Schema Diagnostic: Which schema does Vercel read from?

The dashboard shows **different data** than `alpha_prime.ironforge` tables.
This notebook checks if `alpha_prime.default` has stale data that the webapp might be reading.

Dashboard shows: INFERNO balance=$10,044, P&L=+$44, trades=18, collateral=$9,233
ironforge schema: INFERNO balance=$9,791, P&L=-$209, trades=28, collateral=$0

In [ ]:
# Check if alpha_prime.default schema has IronForge tables
print("=" * 70)
print("SCHEMA CHECK: Does alpha_prime.default have IronForge tables?")
print("=" * 70)

for schema in ["ironforge", "default"]:
    print(f"\n--- Schema: alpha_prime.{schema} ---")
    try:
        tables = spark.sql(f"SHOW TABLES IN alpha_prime.{schema} LIKE 'inferno*'").collect()
        if tables:
            print(f"  Found {len(tables)} inferno tables:")
            for t in tables:
                print(f"    {t.tableName}")
        else:
            print(f"  No inferno tables found")
    except Exception as e:
        print(f"  Schema does not exist or error: {e}")

In [ ]:
# Compare inferno paper_account data across both schemas
print("=" * 70)
print("COMPARE: inferno_paper_account in both schemas")
print("=" * 70)

for schema in ["ironforge", "default"]:
    print(f"\n--- alpha_prime.{schema}.inferno_paper_account ---")
    try:
        rows = spark.sql(f"""
            SELECT id, is_active, dte_mode, starting_capital, current_balance,
                   cumulative_pnl, collateral_in_use, buying_power, total_trades
            FROM alpha_prime.{schema}.inferno_paper_account
            ORDER BY id
        """).collect()
        if rows:
            for r in rows:
                active = 'ACTIVE' if r.is_active else 'inactive'
                print(f"  id={r.id} {active} dte={r.dte_mode} | "
                      f"bal=${float(r.current_balance or 0):,.2f} | "
                      f"coll=${float(r.collateral_in_use or 0):,.2f} | "
                      f"pnl=${float(r.cumulative_pnl or 0):,.2f} | "
                      f"trades={r.total_trades}")
        else:
            print("  (empty table)")
    except Exception as e:
        print(f"  Table not found: {e}")

In [ ]:
# Compare open positions across both schemas
print("=" * 70)
print("COMPARE: Open positions in both schemas")
print("=" * 70)

for schema in ["ironforge", "default"]:
    print(f"\n--- alpha_prime.{schema}.inferno_positions ---")
    try:
        rows = spark.sql(f"""
            SELECT status, dte_mode, COUNT(*) as cnt,
                   COALESCE(SUM(collateral_required), 0) as total_collateral,
                   COALESCE(SUM(realized_pnl), 0) as total_pnl
            FROM alpha_prime.{schema}.inferno_positions
            GROUP BY status, dte_mode
            ORDER BY status, dte_mode
        """).collect()
        if rows:
            for r in rows:
                print(f"  status={r.status} dte={r.dte_mode} | "
                      f"count={r.cnt} | "
                      f"collateral=${float(r.total_collateral):,.2f} | "
                      f"pnl=${float(r.total_pnl):,.2f}")
        else:
            print("  (empty table)")
    except Exception as e:
        print(f"  Table not found: {e}")

In [ ]:
# Specifically check what the dashboard status API query would return in EACH schema
# This simulates the exact queries from status/route.ts
print("=" * 70)
print("SIMULATE: What status API returns per schema (INFERNO / 0DTE)")
print("=" * 70)

for schema in ["ironforge", "default"]:
    print(f"\n--- alpha_prime.{schema} ---")
    try:
        # Live P&L (same query as status route)
        pnl = spark.sql(f"""
            SELECT COALESCE(SUM(realized_pnl), 0) as actual_realized_pnl,
                   COUNT(*) as actual_total_trades
            FROM alpha_prime.{schema}.inferno_positions
            WHERE status IN ('closed', 'expired')
              AND realized_pnl IS NOT NULL
              AND dte_mode = '0DTE'
        """).collect()[0]

        # Live collateral (same query as status route)
        coll = spark.sql(f"""
            SELECT COALESCE(SUM(collateral_required), 0) as actual_collateral
            FROM alpha_prime.{schema}.inferno_positions
            WHERE status = 'open' AND dte_mode = '0DTE'
        """).collect()[0]

        realized_pnl = float(pnl.actual_realized_pnl or 0)
        total_trades = int(pnl.actual_total_trades or 0)
        collateral = float(coll.actual_collateral or 0)
        balance = 10000 + realized_pnl
        bp = balance - collateral

        print(f"  P&L:        ${realized_pnl:,.2f} from {total_trades} trades")
        print(f"  Balance:    ${balance:,.2f}")
        print(f"  Collateral: ${collateral:,.2f}")
        print(f"  Buying pwr: ${bp:,.2f}")

        # Check if these match the dashboard values
        if abs(balance - 10044) < 5 and abs(collateral - 9233) < 5:
            print(f"\n  >>> THIS MATCHES THE DASHBOARD! Vercel is reading from alpha_prime.{schema} <<<")
        elif abs(balance - 9791) < 5 and abs(collateral) < 5:
            print(f"  (This matches the corrected database values)")
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# FIX: If alpha_prime.default has stale data, run the same reconciliation there
FIX_DEFAULT_SCHEMA = True  # Set to True to fix the default schema too

if FIX_DEFAULT_SCHEMA:
    print("=" * 70)
    print("FIXING: alpha_prime.default schema (if it has stale data)")
    print("=" * 70)

    for bot_name, dte in [("flame", "2DTE"), ("spark", "1DTE"), ("inferno", "0DTE")]:
        schema = "default"
        table_prefix = f"alpha_prime.{schema}.{bot_name}"
        print(f"\n--- {bot_name.upper()} ({dte}) in alpha_prime.{schema} ---")

        try:
            # Step 1: Close all open positions (they're stale)
            open_pos = spark.sql(f"""
                SELECT position_id, collateral_required, total_credit, contracts
                FROM {table_prefix}_positions
                WHERE status = 'open'
            """).collect()

            if open_pos:
                print(f"  Closing {len(open_pos)} stale open positions...")
                for pos in open_pos:
                    credit = float(pos.total_credit or 0)
                    # Close at entry credit (no P&L for stale positions)
                    spark.sql(f"""
                        UPDATE {table_prefix}_positions
                        SET status = 'closed',
                            close_time = CURRENT_TIMESTAMP(),
                            close_price = {credit},
                            realized_pnl = 0,
                            close_reason = 'stale_schema_fix',
                            updated_at = CURRENT_TIMESTAMP()
                        WHERE position_id = '{pos.position_id}'
                          AND status = 'open'
                    """)
                    print(f"    Closed: {pos.position_id}")
            else:
                print(f"  No open positions")

            # Step 2: Reconcile paper_account
            pnl = spark.sql(f"""
                SELECT COALESCE(SUM(realized_pnl), 0) as total_pnl,
                       COUNT(*) as total_trades
                FROM {table_prefix}_positions
                WHERE status IN ('closed', 'expired')
                  AND realized_pnl IS NOT NULL
                  AND dte_mode = '{dte}'
            """).collect()[0]

            actual_pnl = float(pnl.total_pnl or 0)
            actual_trades = int(pnl.total_trades or 0)
            expected_balance = round(10000 + actual_pnl, 2)

            print(f"  Reconciling paper_account: balance=${expected_balance:,.2f}, pnl=${actual_pnl:,.2f}, trades={actual_trades}, collateral=$0")

            spark.sql(f"""
                UPDATE {table_prefix}_paper_account
                SET current_balance = {expected_balance},
                    cumulative_pnl = {actual_pnl},
                    collateral_in_use = 0,
                    buying_power = {expected_balance},
                    total_trades = {actual_trades},
                    high_water_mark = GREATEST(high_water_mark, {expected_balance}),
                    updated_at = CURRENT_TIMESTAMP()
                WHERE dte_mode = '{dte}'
            """)
            print(f"  FIXED")

        except Exception as e:
            if "TABLE_OR_VIEW_NOT_FOUND" in str(e) or "does not exist" in str(e).lower():
                print(f"  Table does not exist in this schema (not the issue)")
            else:
                print(f"  Error: {e}")

    print(f"\n{'=' * 70}")
    print("Done. Refresh the dashboard to verify.")
    print("If still wrong, check Vercel env var DATABRICKS_SCHEMA")
    print("=" * 70)